# 🌍 Simple Disaster COG Processing

> **Dev/Testing Version:** This notebook uses Python imports from `shared_utils` for development testing. For production Hub deployment, use the CLI version in `notebooks/`.

This simplified notebook converts disaster satellite imagery to Cloud Optimized GeoTIFFs (COGs) with just a few cells.

## ✨ Features
- **See files first** - List S3 files before configuring
- **Smart configuration** - Define filename functions after seeing actual files
- **Auto-discovery** - Automatically categorizes your files
- **Simple processing** - Just run the cells in order

---

## 📋 Step 1: Basic Configuration

Set your event details and S3 paths:

In [ ]:
# ========================================
# INPUTS
# ========================================

# S3 Paths
BUCKET = 'nasa-disasters'                    # S3 bucket (DO NOT CHANGE)
DESTINATION_BASE = 'drcs_activations_new'    # Where COGs are saved in S3 (DO NOT CHANGE)
GEOTIFF_DIR = 'drcs_activations'             # Where non-converted source files live
DESTINATION_PRODUCT = 'Vantor'               # Cosmetic label only; actual routing is per-category (Step 3 OUTPUT_DIRS)

# Event Details
EVENT_NAME = '202606_Earthquake_Venezuela'   # YYYYMM_Hazard_Location
SOURCE = "Vantor"                            # Data origin (e.g., USGS, Copernicus, CSDA, Vantor, TBD)
SUB_PRODUCT_NAME = 'Vantor'                  # Sub-directory under the event
SOURCE_PATH = f'{GEOTIFF_DIR}/{EVENT_NAME}/{SUB_PRODUCT_NAME}'

# Processing Options
OVERWRITE = False      # True to replace existing files
VERIFY = True          # True to verify results after processing
SAVE_RESULTS = True    # False to skip saving results CSV to /output

# None preserves source CRS (fastest). Uncomment EPSG:3857 for veda-data-airflow build_stac.
TARGET_CRS = None
# TARGET_CRS = "EPSG:3857"

print(f"Source: s3://{BUCKET}/{SOURCE_PATH}")
print(f"Destination base: s3://{BUCKET}/{DESTINATION_BASE}/ (per-category subdirs)")

In [ ]:
from shared_utils import PROCESSOR_STRING
ACTIVATION_METADATA = {
    "ACTIVATION_EVENT": EVENT_NAME,
    "SOURCE": SOURCE,
    "PROCESSOR": PROCESSOR_STRING,
    # Add any custom key-value pairs here
}


## 🔍 Step 2: Connect to S3 and List Files

Let's see what files are available before configuring filename transformations:

In [ ]:
# Import necessary modules
import sys
import os
from pathlib import Path

# Add repo root to path (two levels up from testing-notebooks/)
# Find repo root by walking up until we find shared_utils/
repo_root = Path('.').resolve()
for _ in range(5):
    if (repo_root / 'shared_utils').is_dir():
        break
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

# Import S3 operations
from shared_utils.s3_operations import (
    initialize_s3_client,
    list_s3_files,
    get_file_size_from_s3
)

# Initialize S3 client
print("Connecting to S3...")
s3_client, _ = initialize_s3_client(bucket_name=BUCKET, verbose=False)

if s3_client:
    print("Connected to S3\n")
    
    # List all TIF files
    print(f"Files in s3://{BUCKET}/{SOURCE_PATH}:")
    print("="*60)
    
    files = list_s3_files(s3_client, BUCKET, SOURCE_PATH, suffix='.tif')
    
    if files:
        print(f"Found {len(files)} .tif files:\n")
        for i, file_path in enumerate(files[:10], 1):  # Show first 10
            filename = os.path.basename(file_path)
            try:
                size_gb = get_file_size_from_s3(s3_client, BUCKET, file_path)
                print(f"{i:2}. {filename:<60} ({size_gb:.2f} GB)")
            except:
                print(f"{i:2}. {filename}")
        
        if len(files) > 10:
            print(f"\n... and {len(files) - 10} more files")
        
        print("\n" + "="*60)
        print("\nUse this information to create filename functions in Step 3")
    else:
        print("No .tif files found in the specified path.")
        print("   Check your SOURCE_PATH configuration.")
else:
    print("Could not connect to S3. Check your AWS credentials.")
    files = []

## 🔐 Step 2.5 (Optional): Enable Upload Permissions

By default, S3 authentication uses **read-only** credentials. To enable **upload permissions** for processing files, you need to configure an external ID.

### Why External ID?
The external ID acts like a password that allows you to assume an AWS role with upload permissions. Without it, you can read files but cannot write processed COGs back to S3.

### Setup Instructions:
1. Copy the template file to create your credentials file
2. Add the external ID (get this from your administrator)

Run the cell below to check your current authentication status:

In [ ]:
# Check if external ID is configured for upload permissions
import os
from pathlib import Path

# Find repo root (same as cell 4)
creds_dir = Path('.').resolve()
for _ in range(5):
    if (creds_dir / 'shared_utils').is_dir():
        break
    creds_dir = creds_dir.parent

creds_file = creds_dir / 'aws_credentials.py'
example_file = creds_dir / 'aws_credentials.example.py'

print("Upload Permissions Status Check")
print("="*60)

if creds_file.exists():
    print("aws_credentials.py found")
    
    try:
        sys.path.insert(0, str(creds_dir))
        from aws_credentials import EXTERNAL_ID, UPLOAD_ROLE_ARN
        
        if EXTERNAL_ID and EXTERNAL_ID != "your-external-id-here":
            print("External ID is configured")
            print("You have UPLOAD permissions enabled")
            print(f"\n   Role: {UPLOAD_ROLE_ARN}")
            print("\nYou're all set to upload processed files to S3!")
        else:
            print("External ID not configured (still using placeholder)")
            print("Currently using READ-ONLY credentials")
    except ImportError as e:
        print(f"Could not import credentials: {e}")
        print("Currently using READ-ONLY credentials")
else:
    print("aws_credentials.py not found")
    print("Currently using READ-ONLY credentials")
    print(f"\nTo enable uploads, copy the template:")
    print(f"   cp {example_file} {creds_file}")

print("\n" + "="*60)

### Quick check to see if a file will upload to S3 with credentials and External ID

In [10]:
from shared_utils.test_upload import test_s3_upload
test_s3_upload()

🧪 Testing S3 Upload Permissions

1. Initializing S3 client...
🔑 Attempting to authenticate with external ID for upload permissions...
✅ S3 client initialized with UPLOAD permissions via external ID
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized

2. Creating test file...
   ✅ Created: /tmp/tmprlp4ntvs.txt

3. Uploading to S3...
   [UPLOAD] Uploading 0.0 MB to s3://nasa-disasters/test_uploads/test.txt
   [UPLOAD] ✅ Uploaded to s3://nasa-disasters/test_uploads/test.txt

✅ SUCCESS! File uploaded to s3://nasa-disasters/test_uploads/test.txt

🎉 Upload permissions are working correctly!
   You can now process and upload COG files.

4. Verifying upload...
   ✅ File verified in S3 (120 bytes)

   🧹 Cleaned up temporary file



True

## 🏷️ Step 3a: Define Categorization and Filename Transformations

Based on the files you see above, configure:
1. **Categorization patterns** - Regex patterns to identify file types
2. **Filename functions** - How to transform filenames
3. **Output directories** - Where each category should be saved

In [ ]:
# ========================================
# CATEGORIZATION AND OUTPUT CONFIGURATION
# ========================================
# Pre-wired for the common NASA Disasters product suite. Delete categories you
# don't need for this event, or add a new one by giving it four things: a regex
# in CATEGORIZATION_PATTERNS, a builder in FILENAME_CREATORS, a subdir in
# OUTPUT_DIRS, and (optionally) a value in NODATA_VALUES.

import os
import re

# STEP 1: Pull the acquisition date (YYYYMMDD) out of a filename.
def extract_date_from_filename(filename):
    """Extract date from filename in YYYYMMDD format."""
    dates = re.findall(r'\d{8}', filename)
    if dates:
        date_str = dates[0]
        # return f"{date_str[0:4]}-{date_str[4:6]}-{date_str[6:8]}"  # hyphenated variant
        return f"{date_str}"
    return None

# STEP 2: Per-product filename builders. They share one convention today
# ({event}_{stem_without_date}_{date}_day.tif) but are kept as separate
# functions so an individual product's naming can be changed in isolation.
def create_truecolor_filename(original_path, event_name):
    """Create filename for trueColor products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_colorinfrared_filename(original_path, event_name):
    """Create filename for colorInfrared products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_naturalcolor_filename(original_path, event_name):
    """Create filename for naturalColor products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_swir_filename(original_path, event_name):
    """Create filename for shortwaveIR (swirRGB) products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_ndvi_filename(original_path, event_name):
    """Create filename for ndvi products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_mndwi_filename(original_path, event_name):
    """Create filename for mndwi products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_ndwi_filename(original_path, event_name):
    """Create filename for ndwi products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_waterextent_filename(original_path, event_name):
    """Create filename for waterExtent products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_blackmarble_filename(original_path, event_name):
    """Create filename for blackmarble products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_cloudmask_filename(original_path, event_name):
    """Create filename for cloudMask products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_nbr_filename(original_path, event_name):
    """Create filename for nbr products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_dnbr_filename(original_path, event_name):
    """Create filename for dnbr products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_ssrun_filename(original_path, event_name):
    """Create filename for ssrun products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

def create_psh_filename(original_path, event_name):
    """Create filename for psh (R1C1) products."""
    filename = os.path.basename(original_path)
    stem = os.path.splitext(filename)[0]
    date = extract_date_from_filename(stem)

    if date:
        stem_clean = re.sub(r'_\d{8}', '', stem)
        return f"{event_name}_{stem_clean}_{date}_day.tif"
    return f"{event_name}_{stem}_day.tif"

# STEP 3: Categorization patterns (regex -> category). Files matching no
# pattern are skipped with a warning.
CATEGORIZATION_PATTERNS = {
    'trueColor': r'trueColor|truecolor|true_color|(?:^|[_-])tc(?=[_.-]|$)|trueColorMasked',
    'colorInfrared': r'colorInfrared|colorIR|color_infrared|CIR',
    'naturalColor': r'naturalColor|natural_color|natural',
    'shortwaveIR': r'shortwaveInfrared|swir',
    'ndvi': r'NDVI|ndvi',
    'mndwi': r'mNDWI|mndwi|MNDWI',
    'ndwi': r'NDWI|ndwi',
    'waterExtent': r'WaterExtent|waterextent|water_extent',
    'blackmarble': r'BlackMarble',
    'cloudMask': r'cloudMask|cloudMasks|cloud_mask|cloud_masks',
    'dnbr': r'dnbr|DNBR|dNBR',
    'nbr': r'nbr|NBR',
    'ssrun': r'ssrun|SSRUN',
    'psh': r'R1C1',
}

# STEP 4: category -> filename builder (one entry per category above).
FILENAME_CREATORS = {
    'trueColor': create_truecolor_filename,
    'colorInfrared': create_colorinfrared_filename,
    'naturalColor': create_naturalcolor_filename,
    'shortwaveIR': create_swir_filename,
    'ndvi': create_ndvi_filename,
    'mndwi': create_mndwi_filename,
    'ndwi': create_ndwi_filename,
    'waterExtent': create_waterextent_filename,
    'blackmarble': create_blackmarble_filename,
    'cloudMask': create_cloudmask_filename,
    'nbr': create_nbr_filename,
    'dnbr': create_dnbr_filename,
    'ssrun': create_ssrun_filename,
    'psh': create_psh_filename,
}

# STEP 5: category -> output subdirectory, RELATIVE to the S3 destination base
# (the processor prepends DESTINATION_BASE). Keep these as plain subdirs (no
# leading base var) so an empty value can't produce a "//" key on S3.
OUTPUT_DIRS = {
    'trueColor': 'trueColor',
    'colorInfrared': 'colorIR',
    'naturalColor': 'naturalColor',
    'shortwaveIR': 'shortwaveIR',
    'ndvi': 'NDVI',
    'mndwi': 'MNDWI',
    'ndwi': 'NDWI',
    'waterExtent': 'waterExtent',
    'blackmarble': 'BlackMarble',
    'cloudMask': 'cloudMask',
    'nbr': 'NBR',
    'dnbr': 'dNBR',
    'ssrun': 'SSRUN',
    'psh': 'PSH',
}

# OPTIONAL: category -> nodata value (None / missing = auto-detect from dtype).
NODATA_VALUES = {
    'trueColor': 0,
    'colorInfrared': 0,
    'naturalColor': 0,
    'shortwaveIR': 0,
    'ndvi': -9999,
    'mndwi': -9999,
    'ndwi': -9999,
    'waterExtent': -9999,
    'blackmarble': -9999,
    'cloudMask': 0,
    'nbr': -9999,
    'dnbr': -9999,
    'ssrun': 9.9990003E+20,
    'psh': 1,
}


## 🏷️ Step 3b: Test the new functions to verify what the inputs and outputs will be. 

In [ ]:
print("✅ Configuration defined")
print(f"\n📂 Categories and output paths:")
for category, path in OUTPUT_DIRS.items():
    pattern = CATEGORIZATION_PATTERNS.get(category, 'No pattern defined')
    print(f"   • {category}:")
    print(f"     Pattern: {pattern}")
    print(f"     Output:  {DESTINATION_BASE}/{path}")

# Test with sample filename if files exist
if files:
    sample_file = files[0]
    sample_name = os.path.basename(sample_file)
    
    # Check which category it would match
    matched_category = None
    for cat, pattern in CATEGORIZATION_PATTERNS.items():
        if re.search(pattern, sample_name, re.IGNORECASE):
            matched_category = cat
            break
    
    if matched_category:
        new_name = FILENAME_CREATORS[matched_category](sample_file, EVENT_NAME)
        print(f"\n📝 Example transformation:")
        print(f"   Original: {sample_name}")
        print(f"   Category: {matched_category}")
        print(f"   → New:    {new_name}")
        print(f"   → Output: {DESTINATION_BASE}/{OUTPUT_DIRS[matched_category]}/{new_name}")
    else:
        print(f"\n⚠️ Warning: Sample file doesn't match any category pattern:")
        print(f"   File: {sample_name}")
        print(f"   Add a pattern to CATEGORIZATION_PATTERNS to process this file type")

## 🚀 Step 4: Initialize Processor and Preview

Now let's set up the processor and preview all transformations:

In [ ]:
# Import our simplified helper
from shared_utils.notebook_helpers import SimpleProcessor

# Create full configuration with categorization patterns
config = {
    'event_name': EVENT_NAME,
    'bucket': BUCKET,
    'source_path': SOURCE_PATH,
    'destination_base': DESTINATION_BASE,
    'overwrite': OVERWRITE,
    'verify': VERIFY,
    'save_results': SAVE_RESULTS,  # Add save results flag
    'categorization_patterns': CATEGORIZATION_PATTERNS,  # IMPORTANT: Include patterns
    'filename_creators': FILENAME_CREATORS,
    'output_dirs': OUTPUT_DIRS,
    'nodata_values': NODATA_VALUES,
    'target_crs': TARGET_CRS,  # None preserves source CRS; 'EPSG:3857' to reproject for veda-data-airflow
    'metadata': ACTIVATION_METADATA,  # Embedded as GeoTIFF tags on every output COG
}

# Initialize processor
processor = SimpleProcessor(config)

# Connect to S3 (already connected, but needed for processor)
if processor.connect_to_s3():
    print("✅ Processor ready\n")
    
    # Discover and categorize files
    num_files = processor.discover_files()
    
    if num_files > 0:
        # Show preview of transformations
        processor.preview_processing()
        
        print("\n📌 Review the transformations above.")
        print("   • Files will be saved to the directories specified in OUTPUT_DIRS")
        print("   • If files appear as 'uncategorized', add patterns to CATEGORIZATION_PATTERNS")
        print("   • When ready, proceed to Step 5 to process the files.")
    else:
        print("⚠️ No files found to process.")
else:
    print("❌ Could not initialize processor.")

## ⚙️ Step 5: Process Files

Run this cell to start processing all files:

In [ ]:
# Process all files
if 'num_files' in locals() and num_files > 0:
    print("🚀 Starting processing...")
    print("This may take several minutes depending on file sizes.\n")
    
    # Process everything
    results = processor.process_all()
    
    # Display results
    if not results.empty:
        print("\n📊 Processing Complete!")
        display(results) if 'display' in dir() else print(results)
else:
    print("⚠️ No files to process. Complete Steps 1-4 first.")

In [ ]:
# Analyze results
if 'results' in locals() and not results.empty:
    print("📊 PROCESSING STATISTICS")
    print("="*40)
    
    # Success rate
    total = len(results)
    success = len(results[results['status'] == 'success'])
    failed = len(results[results['status'] == 'failed'])
    skipped = len(results[results['status'] == 'skipped'])
    
    print(f"Total files: {total}")
    print(f"✅ Success: {success}")
    print(f"❌ Failed: {failed}")
    print(f"⏭️ Skipped: {skipped}")
    print(f"\nSuccess rate: {(success/total*100):.1f}%")
    
    # Failed files
    if failed > 0:
        print("\n❌ Failed files:")
        failed_df = results[results['status'] == 'failed']
        for idx, row in failed_df.iterrows():
            print(f"  - {row['source_file']}: {row.get('error', 'Unknown error')}")
    
    # Processing times
    if 'time_seconds' in results.columns:
        success_df = results[results['status'] == 'success']
        if not success_df.empty:
            avg_time = success_df['time_seconds'].mean()
            max_time = success_df['time_seconds'].max()
            print(f"\n⏱️ Timing:")
            print(f"Average: {avg_time:.1f} seconds per file")
            print(f"Slowest: {max_time:.1f} seconds")
else:
    print("No results to analyze. Run Step 5 first.")

## 💡 Tips & Troubleshooting

### Workflow Summary:
1. **Setup** - Clone disasters-aws-conversion repository (Step 0)
2. **Configure** basic settings (Step 1)
3. **List files** from S3 to see naming patterns (Step 2)
4. **Define functions** to transform filenames (Step 3)
5. **Preview** transformations (Step 4)
6. **Process** all files (Step 5)
7. **Review** results (Step 6)

### Common Issues:

1. **"ModuleNotFoundError: No module named 'core'" or import errors**
   - Run Step 0 first to clone the disasters-aws-conversion repository
   - Restart kernel and run all cells from the beginning

2. **"No files found"**
   - Check `SOURCE_PATH` in Step 1
   - Verify bucket permissions
   - Ensure files have `.tif` extension

3. **Wrong filename transformations**
   - Review actual filenames in Step 2
   - Adjust functions in Step 3
   - Re-run Step 4 to preview

4. **Files being skipped**
   - Files already exist in destination
   - Set `OVERWRITE = True` in Step 1

5. **Processing errors**
   - Check AWS credentials
   - Verify S3 write permissions
   - Check available disk space for temp files

### Need More Control?

Use the full template at `disaster_processing_template.ipynb` for:
- Manual chunk configuration
- Custom compression settings
- Detailed memory management
- Advanced processing options

In [ ]:
# Preview a sample of the generated COGs to sanity-check the processing visually.
# preview_cogs reads via rasterio with overview-aware downsampling so it stays
# fast even on multi-GB COGs.
from shared_utils.plotting import preview_cogs

# Pull successful output paths from the results list. Each notebook's worker
# stashes the COG path under one of these keys; we look at all of them.
_successful = [r for r in results if r.get('status') == 'success']
_paths = []
for r in _successful:
    for key in ('output_path', 'cog_path', 'local_cog', 'output_file'):
        v = r.get(key)
        if v and isinstance(v, str) and v.endswith('.tif'):
            _paths.append(v)
            break

preview_cogs(_paths, sample_n=4)
